In [ ]:
import json
from typing import Optional

In [ ]:
from dotenv import load_dotenv
import os


load_dotenv()

OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
OPEN_ROUTER_COMPLETION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
OPEN_ROUTER_URL = "https://openrouter.ai/api/v1"

OLLAMA_COMPLETION_MODEL = "gemma4:e4b"


OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
OLLAMA_API_KEY='ollama'


OPEN_ROUTER_HEADERS = {
        "HTTP-Referer": "https://github.com/pixelbuildlab/be-story-teller",
        "X-OpenRouter-Title": "Be story teller - Agentic way to stories",
    }

In [ ]:
USE_OLLAMA=True
MODEL= OLLAMA_COMPLETION_MODEL if USE_OLLAMA else OPEN_ROUTER_COMPLETION_MODEL
API_URL= OLLAMA_API_URL if USE_OLLAMA else OPEN_ROUTER_URL
API_KEY = OLLAMA_API_KEY if USE_OLLAMA else OPEN_ROUTER_API_KEY
HEADERS = None if USE_OLLAMA else OPEN_ROUTER_HEADERS

In [ ]:
USE_OLLAMA, MODEL, API_URL,  HEADERS

In [ ]:
import openai
from openai import OpenAI

client = OpenAI(
    base_url=API_URL,
    api_key=API_KEY,
    default_headers=HEADERS,
)


In [ ]:
SYSTEM_META_PROMPT = """
You are a professional children's story writer.

Your goals:
- Write bedtime stories for children aged 4–9.
- Stories should be calming.
- Never include violence or horror.
- Keep language simple.
- If you need to use a tool, use it.
- Stories should be relaxing, pleasing to hear and lesson full.
"""

In [ ]:
def AI(messages: list, tools: Optional[list] | None):
    chat_completion = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    return chat_completion

In [ ]:
META_PROMPT_OPTIMIZER = """
You are an expert prompt optimizer for children's story generation.

Your task is to transform short or incomplete user requests into rich, detailed prompts for a story-writing AI.

Rules:
- Preserve the user's original intent.
- Add reasonable assumptions when details are missing.
- Specify:
  - protagonist
  - setting
  - conflict
  - tone
  - target age
  - ending
  - approximate length
- Do not write the story.
- Return ONLY the optimized prompt.
"""


async def stroy_optimizer_tool(prompt: str):
    print("starting stroy_optimizer_tool")
    messages = [
        {"role": "system", "content": META_PROMPT_OPTIMIZER},
        {
            "role": "user",
            "content": f"{prompt}",
        },
    ]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("stroy_optimizer_tool called")
    return response_message, chat_completion

In [ ]:
META_PROMPT_PLANNER = """
You are a professional children's story planner.

Your responsibility is to create a structured plan for a children's bedtime story.

The final story will be written by another AI, so only create the plan.

Story Requirements:
- Audience: Children aged 4–9.
- Genre: Bedtime story.
- Tone: Calm, gentle, comforting and relaxing.
- Language should remain simple.
- The story must teach a positive lesson or moral.
- Never include violence, horror, frightening scenes, or inappropriate themes.
- The story should be enjoyable to listen to before bedtime.

Generate ONLY valid JSON using the following structure:

{
    "title": "",
    "genre": "Children's Bedtime",
    "tone": "",
    "premise": "",
    "lesson": "",
    "ending": "",
    "characters": [
        {
            "name": "",
            "role": "",
            "personality": ""
        }
    ],
    "outline": [
        {
            "id": 1,
            "summary": "",
            "goal": ""
        }
    ]
}

Planning Guidelines:
- Create 3–5 outline sections.
- Introduce the main character in the first section.
- Present a small, age-appropriate challenge.
- Resolve the challenge peacefully.
- End with a satisfying, comforting conclusion.
- Ensure the lesson naturally emerges from the story.
- Keep every event suitable for children aged 4–9.
- If the user provides story details, preserve them.
- Otherwise, invent wholesome and creative details.

Rules:
- Return ONLY valid JSON.
- Do not write the actual story.
- Do not explain your reasoning.
- Do not include markdown.
"""

async def story_outline_planner_tool(optimized_prompt:str):
    # create an outline based on the system
    # this will help AI to be aligned with a single path
    # donot add characters and stray time waste
    print("Started story_outline_planner_tool")
    messages = [
        {"role": "system", "content": META_PROMPT_PLANNER},
        {
            "role": "user",
            "content": f"{optimized_prompt}",
        },
    ]

    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("End story_outline_planner_tool")
    return response_message, chat_completion

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "stroy_optimizer_tool",
            "description": "Optimize user input prompt to a level it creates stunning storyline. Not required if input prompt is valid and meaningful for a story",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "User input prompt to optimize",
                    }
                },
                "required": ["prompt"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "story_outline_planner_tool",
            "description": "Work on storyline, scenes and characters for the story. It outputs a formatted JSON for storyline planning.",
            "parameters": {
                "type": "object",
                "properties": {
                    "optimized_prompt": {
                        "type": "string",
                        "description": "Optimized prompt to draft a storyline",
                    }
                },
                "required": ["optimized_prompt"],
            },
        },
    },
]

In [ ]:
class AgentService:
    MESSAGES_LIST = []   
    async def agent_flow(self, prompt: str, META_PROMPT: str):
        try:
            messages = [
                {"role": "system", "content": META_PROMPT},
                {
                    "role": "user",
                    "content": f"{prompt}",
                },
            ]
            self.MESSAGES_LIST.extend(messages)

            while True:
                print("STARTING AGENT")
                chat_completion = AI(self.MESSAGES_LIST, tools)
                response_message = chat_completion.choices[0].message

                self.MESSAGES_LIST.append(response_message.model_dump())
                print(f"MAIN chat output: {response_message}")

                # If LLM returned tool calls, process them
                if hasattr(response_message, "tool_calls") and response_message.tool_calls:
                    for tool_call in response_message.tool_calls:
                        function_name = tool_call.function.name
                        function_args = json.loads(tool_call.function.arguments)

                        print(f"Tool call: {function_name}, args: {function_args}")
                        agent_args = []

                        tool_function = globals()[function_name]

                        tool_result, tool_chat_completion = await tool_function(
                            *agent_args, **function_args
                        )

                        self.MESSAGES_LIST.append(
                            {
                                "role": "tool",
                                "tool_name": function_name,
                                "tool_call_id": tool_call.id,
                                "content": tool_result.content,
                            }
                        )

                else:
                    # LIKELY TO STOP LOOP
                    # IF BUGGY USE A TOOL TO SOP THE LOOP.
                    return chat_completion

                # return chat_completion

        except openai.APIConnectionError as e:
            print(f"Network connectivity issue: {e}")
        except openai.RateLimitError as e:
            print(f"Rate limits hit or out of funds: {e}")
        except openai.APIStatusError as e:
            print(f"HTTP Error received (Status: {e.status_code}): {e.response}")

In [ ]:
agent =  AgentService()


In [ ]:
await agent.agent_flow('A friend in need is a friend indeed', SYSTEM_META_PROMPT)

In [ ]:
agent.MESSAGES_LIST